# Việt Phục Remix — Server GPU CatVTON Thật (Khóa Mặt 100% • Giữ Đúng Tỷ Lệ)

**Công nghệ:** `zhengchong/CatVTON` + `SegFormer B2 Clothes` (tự tách vùng quần áo, **tuyệt đối không chạm mặt/tóc/nền**) + `Letterbox Pad/Unpad` (không bao giờ ép vuông hay phóng to ảnh).

**3 bước chạy sáng hôm thi (0đ trên Google Colab T4):**
1. Menu **Runtime → Change runtime type → T4 GPU** → **Save**.
2. Bấm **Runtime → Run all** (lần đầu tải model khoảng 3–5 phút).
3. Copy link `https://xxxx.trycloudflare.com` ở Cell 3 → dán vào **Cài đặt Demo** trên tab **AI Thử Ảnh** của web.

In [ ]:
# Cell 1: Kiểm tra GPU T4, clone CatVTON & cài thư viện (khóa đúng phiên bản tương thích)
!nvidia-smi
!rm -rf /content/CatVTON && GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://huggingface.co/spaces/zhengchong/CatVTON /content/CatVTON
!ls -la /content/CatVTON/model
!pip install -q fastapi uvicorn pillow diffusers==0.27.2 huggingface_hub==0.25.2 transformers accelerate safetensors fvcore omegaconf einops
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
print('Cài đặt hoàn tất!')

In [ ]:
# Cell 2: Tải CatVTONPipeline (fp16) + SegFormer Cloth Parser (tự khoanh đúng quần áo, khóa cứng mặt & tóc)
import os, sys, io, base64, subprocess, torch, numpy as np
from PIL import Image, ImageFilter, ImageOps
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
from huggingface_hub import snapshot_download

if not os.path.exists('/content/CatVTON/model/pipeline.py'):
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(['rm', '-rf', '/content/CatVTON'], check=True)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://huggingface.co/spaces/zhengchong/CatVTON', '/content/CatVTON'], check=True, env=env)
os.chdir('/content/CatVTON')
if '/content/CatVTON' not in sys.path:
    sys.path.insert(0, '/content/CatVTON')
from model.pipeline import CatVTONPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32

# 1. SegFormer nhận diện chính xác pixel quần áo (label 4=Upper, 5=Skirt, 6=Pants, 7=Dress, 8=Belt, 17=Scarf)
# Tuyệt đối bảo vệ label 1=Hat, 2=Hair, 11=Face, 0=Background
seg_processor = SegformerImageProcessor.from_pretrained('mattmdjaga/segformer_b2_clothes')
seg_model = AutoModelForSemanticSegmentation.from_pretrained('mattmdjaga/segformer_b2_clothes').to(device).eval()

# 2. Tải weights CatVTON
repo_path = snapshot_download(repo_id='zhengchong/CatVTON')
catvton_pipe = CatVTONPipeline(
    base_ckpt='booksforcharlie/stable-diffusion-inpainting',
    attn_ckpt=repo_path,
    attn_ckpt_version='mix',
    weight_dtype=dtype,
    use_tf32=True,
    device=device
)
print('CatVTON + SegFormer đã sẵn sàng trên:', device)

In [ ]:
# Cell 3: FastAPI Server (/health, /try-on) với Letterbox Pad/Unpad & Khóa Mặt 100% + Cloudflare Tunnel
import re, subprocess, threading, time
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

app = FastAPI(title='VietPhucRemix CatVTON GPU Server')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*']
)

class TryOnReq(BaseModel):
    personImageDataUrl: str
    garmentImageDataUrl: str | None = None
    garmentId: str = 'ngu-than'
    prompt: str = ''

def decode_data_url(data_url: str) -> Image.Image:
    _, b64 = data_url.split(',', 1)
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGB')

def encode_png_data_url(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode('utf-8')

def letterbox_pad(img: Image.Image, target_w=768, target_h=1024, fill=(245, 240, 232)):
    ow, oh = img.size
    scale = min(target_w / ow, target_h / oh)
    nw, nh = int(round(ow * scale)), int(round(oh * scale))
    resized = img.resize((nw, nh), Image.LANCZOS)
    canvas = Image.new('RGB', (target_w, target_h), fill)
    px = (target_w - nw) // 2
    py = (target_h - nh) // 2
    canvas.paste(resized, (px, py))
    return canvas, (px, py, nw, nh, ow, oh)

def letterbox_unpad(canvas: Image.Image, meta):
    px, py, nw, nh, ow, oh = meta
    cropped = canvas.crop((px, py, px + nw, py + nh))
    return cropped.resize((ow, oh), Image.LANCZOS)

def build_cloth_mask(person_768: Image.Image, garment_id: str) -> Image.Image:
    inputs = seg_processor(images=person_768, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = seg_model(**inputs).logits
    upsampled = torch.nn.functional.interpolate(
        logits, size=(1024, 768), mode='bilinear', align_corners=False
    )
    seg = upsampled.argmax(dim=1)[0].cpu().numpy()

    # Áo Bà Ba chỉ thay phần áo trên; các dòng cổ phục tà dài thay cả áo + quần/váy
    cloth_labels = [4, 7, 8, 17] if garment_id == 'ao-ba-ba' else [4, 5, 6, 7, 8, 17]
    cloth_bool = np.isin(seg, cloth_labels)
    # Mở rộng nhẹ mask quần áo để ôm đủ tay thụng nhưng TRỪ TUYỆT ĐỐI mặt (11), tóc (2), mũ (1)
    mask_img = Image.fromarray((cloth_bool.astype(np.uint8) * 255), mode='L')
    mask_img = mask_img.filter(ImageFilter.MaxFilter(25))
    mask_np = np.array(mask_img)
    protect_bool = np.isin(seg, [1, 2, 11])
    protect_img = Image.fromarray((protect_bool.astype(np.uint8) * 255), mode='L').filter(ImageFilter.MaxFilter(11))
    mask_np[np.array(protect_img) > 0] = 0
    return Image.fromarray(mask_np, mode='L')

@app.get('/health')
def health():
    return {'ok': True, 'engine': 'CatVTON+SegFormer', 'device': device}

@app.post('/try-on')
def try_on(req: TryOnReq):
    import traceback
    try:
        if not req.garmentImageDataUrl:
            return {'error': 'Thiếu ảnh mẫu Việt phục (garmentImageDataUrl).'}

        orig_person = decode_data_url(req.personImageDataUrl)
        orig_garment = decode_data_url(req.garmentImageDataUrl)

        person_768, meta = letterbox_pad(orig_person, 768, 1024)
        garment_768, _ = letterbox_pad(orig_garment, 768, 1024, fill=(255, 255, 255))
        mask_768 = build_cloth_mask(person_768, req.garmentId)

        generator = torch.Generator(device=device).manual_seed(42)
        result_768 = catvton_pipe(
            image=person_768,
            condition_image=garment_768,
            mask=mask_768,
            num_inference_steps=22,
            guidance_scale=2.5,
            height=1024,
            width=768,
            generator=generator
        )[0]

        # Khóa cứng 100% khuôn mặt, tóc & nền gốc bằng soft-feather alpha blend
        soft_mask = mask_768.filter(ImageFilter.GaussianBlur(radius=6))
        blended_768 = Image.composite(result_768, person_768, soft_mask)
        final_img = letterbox_unpad(blended_768, meta)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return {'imageUrl': encode_png_data_url(final_img), 'provider': 'colab-catvton'}
    except Exception as e:
        traceback.print_exc()
        return {'error': f'Lỗi trên Colab GPU: {e}'}

threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True
).start()
time.sleep(2)

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('Đang mở Cloudflare Tunnel...')
for line in proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n====================================================')
        print('COPY LINK NÀY DÁN VÀO WEB VIỆT PHỤC REMIX:')
        print(m.group(0))
        print('====================================================\n')
        break

# Tiếp tục xả log cloudflared ở nền để ống pipe không bao giờ bị đầy/treo
threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
